In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import warnings
warnings.filterwarnings("ignore")

base_path = './split/'
fold_path = './split/folds/' # Buat folder khusus

if not os.path.exists(fold_path):
    os.makedirs(fold_path)

# 1. LOAD DATA MENTAH
train_df = pd.read_csv(base_path + '90training.csv')
test_df  = pd.read_csv(base_path + '9testing.csv')

kategori = ['Kabupaten/Kota'] if 'Kabupaten/Kota' in train_df.columns else ['Kabupaten']

# 2. HAPUS & SAMAKAN FORMAT PERIODE
# Hapus kolom Periode di data training karena sudah ada Tahun dan Bulan
if 'Periode' in train_df.columns:
    train_df.drop(columns=['Periode'], inplace=True)

# Ekstrak Tahun dan Bulan di data testing (jika belum ada)
bulan_map = {'Januari':1, 'Februari':2, 'Maret':3, 'April':4, 'Mei':5, 'Juni':6, 
             'Juli':7, 'Agustus':8, 'September':9, 'Oktober':10, 'November':11, 'Desember':12}
if 'Periode' in test_df.columns:
    test_df[['Nama_Bulan', 'Tahun']] = test_df['Periode'].str.split(' ', expand=True)
    test_df['Bulan'] = test_df['Nama_Bulan'].map(bulan_map)
    test_df['Tahun'] = test_df['Tahun'].astype(int)
    test_df.drop(columns=['Periode', 'Nama_Bulan'], inplace=True)

# Bersihkan angka (koma ke titik) untuk kolom numerik
def bersihkan_angka(df):
    for col in df.columns:
        if df[col].dtype == 'object' and col not in kategori:
            df[col] = df[col].astype(str).str.strip().str.replace(',', '.').astype(float)
    return df

train_df = bersihkan_angka(train_df)
test_df = bersihkan_angka(test_df)

# 3. TRANSFORMASI SIN-COS
for df in [train_df, test_df]:
    if 'Bulan' in df.columns:
        df['Bulan_sin'] = np.sin(2 * np.pi * df['Bulan'] / 12)
        df['Bulan_cos'] = np.cos(2 * np.pi * df['Bulan'] / 12)
        df.drop('Bulan', axis=1, inplace=True)

# 4. PROSES 10-FOLD SPLIT, OHE, & NORMALISASI
kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold = 1

print("--- Memulai Proses Pembagian 10-Fold ---")
for train_idx, val_idx in kf.split(train_df):
    # Potong data
    df_tr_fold = train_df.iloc[train_idx].copy().reset_index(drop=True)
    df_val_fold = train_df.iloc[val_idx].copy().reset_index(drop=True)
    
    # A. Proses OHE
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_tr = encoder.fit_transform(df_tr_fold[kategori])
    encoded_val = encoder.transform(df_val_fold[kategori])
    encoded_cols = encoder.get_feature_names_out(kategori)
    
    df_tr_fold = pd.concat([df_tr_fold.drop(columns=kategori), pd.DataFrame(encoded_tr, columns=encoded_cols)], axis=1)
    df_val_fold = pd.concat([df_val_fold.drop(columns=kategori), pd.DataFrame(encoded_val, columns=encoded_cols)], axis=1)
    
    # Simpan target (y) dan paksa (cast) fitur (X) jadi float64 untuk menghindari error sisa
    y_tr_fold = df_tr_fold['Produksi'].values.astype('float64')
    y_val_fold = df_val_fold['Produksi'].values.astype('float64')
    X_tr_fold = df_tr_fold.drop(columns=['Produksi']).values.astype('float64')
    X_val_fold = df_val_fold.drop(columns=['Produksi']).values.astype('float64')
    col_names = df_tr_fold.drop(columns=['Produksi']).columns
    
    # B. Proses Normalisasi Min-Max
    scaler_x = MinMaxScaler()
    X_tr_scaled = scaler_x.fit_transform(X_tr_fold)
    X_val_scaled = scaler_x.transform(X_val_fold)
    
    # Gabungkan kembali X dan y untuk disimpan ke CSV
    final_tr_fold = pd.DataFrame(X_tr_scaled, columns=col_names)
    final_tr_fold['Produksi'] = y_tr_fold
    final_val_fold = pd.DataFrame(X_val_scaled, columns=col_names)
    final_val_fold['Produksi'] = y_val_fold
    
    # Simpan sebagai file CSV fisik
    final_tr_fold.to_csv(f"{fold_path}fold_{fold}_train.csv", index=False)
    final_val_fold.to_csv(f"{fold_path}fold_{fold}_val.csv", index=False)
    
    # Menampilkan informasi jumlah data
    jml_train = len(final_tr_fold)
    jml_val = len(final_val_fold)
    print(f"File Fold {fold} berhasil dibuat. (Train: {jml_train} baris | Val: {jml_val} baris)")
    
    fold += 1

# 5. BUAT FILE FINAL UNTUK PENGUJIAN 90% vs 10%
print("\n--- Memproses File Final (90% vs 10%) ---")
encoder_final = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
enc_tr_final = encoder_final.fit_transform(train_df[kategori])
enc_ts_final = encoder_final.transform(test_df[kategori])
cols_final = encoder_final.get_feature_names_out(kategori)

train_final = pd.concat([train_df.drop(columns=kategori), pd.DataFrame(enc_tr_final, columns=cols_final)], axis=1)
test_final = pd.concat([test_df.drop(columns=kategori), pd.DataFrame(enc_ts_final, columns=cols_final)], axis=1)

scaler_final = MinMaxScaler()
cols_x = train_final.drop(columns=['Produksi']).columns

# Casting ke float64 sebelum scaler
X_tr_final_scaled = scaler_final.fit_transform(train_final.drop(columns=['Produksi']).values.astype('float64'))
X_ts_final_scaled = scaler_final.transform(test_final.drop(columns=['Produksi']).values.astype('float64'))

df_train_ready = pd.DataFrame(X_tr_final_scaled, columns=cols_x)
df_train_ready['Produksi'] = train_final['Produksi'].values
df_test_ready = pd.DataFrame(X_ts_final_scaled, columns=cols_x)
df_test_ready['Produksi'] = test_final['Produksi'].values

df_train_ready.to_csv(base_path + '90training_siap_final.csv', index=False)
df_test_ready.to_csv(base_path + '10testing_siap_final.csv', index=False)

print(f"File Final berhasil dibuat. (Train Final: {len(df_train_ready)} baris | Test Final: {len(df_test_ready)} baris)")
print("\nSeluruh data siap! Lanjut ke Script 2.")

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import joblib
import warnings
import matplotlib.pyplot as plt
import time

warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA & SCALING
# ==========================================
base_path = './split/'
fold_path = './split/folds/'

print(">>> Memuat data 10-Fold dan menginisiasi MinMaxScaler...")
# Load 10 Fold
folds_data = []
for i in range(1, 11):
    tr_df = pd.read_csv(f"{fold_path}fold_{i}_train.csv")
    val_df = pd.read_csv(f"{fold_path}fold_{i}_val.csv")

    X_tr = tr_df.drop(columns=['Produksi']).values
    y_tr_asli = tr_df['Produksi'].values.reshape(-1, 1)

    X_val = val_df.drop(columns=['Produksi']).values
    y_val_asli = val_df['Produksi'].values.reshape(-1, 1)

    # 1. Normalisasi target Produksi dengan MinMaxScaler per fold
    scaler_y_fold = MinMaxScaler()
    y_tr_scaled = scaler_y_fold.fit_transform(y_tr_asli).ravel()
    
    # y_val_scaled disiapkan jika diperlukan, namun evaluasi menggunakan y_val_asli
    y_val_scaled = scaler_y_fold.transform(y_val_asli).ravel() 

    # Simpan scaler dan data asli untuk keperluan inverse_transform
    folds_data.append((X_tr, y_tr_scaled, X_val, y_val_scaled, scaler_y_fold, y_val_asli.ravel()))

# 2. PSO optimasi pada 5-Fold pertama
folds_data_pso = folds_data[:5]

# Final train-test
train_final = pd.read_csv(base_path + '90training_siap_final.csv')
test_final = pd.read_csv(base_path + '10testing_siap_final.csv')

X_train_final = train_final.drop(columns=['Produksi']).values
y_train_final_asli = train_final['Produksi'].values.reshape(-1, 1)

X_test_final = test_final.drop(columns=['Produksi']).values
y_test_final_asli = test_final['Produksi'].values.reshape(-1, 1)

# Scaler global untuk model final
scaler_y_final = MinMaxScaler()
y_train_final_scaled = scaler_y_final.fit_transform(y_train_final_asli).ravel()
y_test_final_scaled = scaler_y_final.transform(y_test_final_asli).ravel()

# Data asli untuk export
test_asli = pd.read_csv(base_path + '9testing.csv')

bulan_map = {
    'Januari':1, 'Februari':2, 'Maret':3, 'April':4,
    'Mei':5, 'Juni':6, 'Juli':7, 'Agustus':8,
    'September':9, 'Oktober':10, 'November':11, 'Desember':12
}

if 'Periode' in test_asli.columns:
    test_asli[['Nama_Bulan', 'Tahun']] = test_asli['Periode'].str.split(' ', expand=True)
    test_asli['Bulan'] = test_asli['Nama_Bulan'].map(bulan_map)
    test_asli['Tahun'] = test_asli['Tahun'].astype(int)
    test_asli.drop(columns=['Periode', 'Nama_Bulan'], inplace=True)

# ==========================================
# 2. FITNESS PSO (5 FOLD)
# ==========================================
def evaluate_svr_cv(params):

    C, epsilon, gamma = params
    rmse_scores = []

    try:
        # Loop hanya pada 5 fold pertama
        for X_tr, y_tr_scaled, X_val, y_val_scaled, scaler_y_fold, y_val_asli in folds_data_pso:

            # Batasan iterasi ditambahkan agar SVR tidak stuck pada parameter buruk
            model = SVR(
                kernel='rbf',
                C=C,
                epsilon=epsilon,
                gamma=gamma,
                max_iter=10000 
            )

            # Training menggunakan target yang dinormalisasi (0-1)
            model.fit(X_tr, y_tr_scaled)

            # Prediksi (masih berskala 0-1)
            y_pred_scaled = model.predict(X_val)

            # Kembalikan ke skala asli (Ton) untuk menghitung error
            y_pred_asli = scaler_y_fold.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

            # Mencegah nilai prediksi negatif 
            y_pred_asli = np.clip(y_pred_asli, 0, None)

            rmse = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli))
            rmse_scores.append(rmse)

        return np.mean(rmse_scores)

    except Exception:
        return float('inf')

# ==========================================
# 3. PSO ALGORITHM (DENGAN TIMEOUT & LOG)
# ==========================================
def pso_auto_resume(n_particles, target_iter, timeout=30):

    np.random.seed(42)

    checkpoint = f"pso_state_rmse_5fold_{n_particles}_partikel.save"

    lb = np.array([1, 0.0001, 0.0001])
    ub = np.array([500, 0.05, 10])

    w, c1, c2 = 0.7, 1.5, 1.5

    if os.path.exists(checkpoint):
        print(f"\n>>> File checkpoint ditemukan! Memuat data {n_particles} partikel...")
        state = joblib.load(checkpoint)

        start_iter = state['iterasi_terakhir']
        particles = state['particles']
        velocities = state['velocities']
        personal_best = state['personal_best']
        personal_best_score = state['personal_best_score']
        global_best = state['global_best']
        global_best_score = state['global_best_score']
        rmse_history = state['rmse_history']
        
        print(f">>> Melanjutkan dari iterasi ke-{start_iter + 1} menuju {target_iter}...\n")
    else:
        print(f"\n>>> Memulai PSO dari awal untuk {n_particles} partikel...\n")
        start_iter = 0
        particles = np.random.uniform(lb, ub, (n_particles, 3))
        velocities = np.zeros((n_particles, 3))
        personal_best = particles.copy()
        personal_best_score = np.array([float('inf')] * n_particles)
        global_best = None
        global_best_score = float('inf')
        rmse_history = []

    if start_iter >= target_iter:
        print("Target iterasi sudah tercapai di run sebelumnya.")
        return rmse_history

    failed_count = 0

    for i in range(start_iter, target_iter):
        print(f"Iterasi ke {i+1}/{target_iter} :")
        iter_success = False

        # Eksekusi partikel secara sekuensial agar bisa mencetak waktu dan skor
        for j in range(n_particles):
            start_time = time.time()
            
            score = evaluate_svr_cv(particles[j])
            
            elapsed = time.time() - start_time
            
            # Jika komputasi melebihi batas timeout
            if elapsed > timeout:
                score = float('inf')

            if score < personal_best_score[j]:
                personal_best[j] = particles[j]
                personal_best_score[j] = score

                if score < global_best_score:
                    global_best = particles[j].copy()
                    global_best_score = score
                    iter_success = True

            score_tampil = f"{score:.4f}" if score != float('inf') else "inf"
            print(f"partikel {j+1}/{n_particles}, RMSE : {score_tampil}, waktu: {elapsed:.4f} s.")

        if not iter_success and global_best_score == float('inf'):
            failed_count += 1
        else:
            failed_count = 0

        rmse_history.append(global_best_score)

        for j in range(n_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            gb = global_best if global_best is not None else personal_best[j]
            
            velocities[j] = (
                w * velocities[j]
                + c1 * r1 * (personal_best[j] - particles[j])
                + c2 * r2 * (gb - particles[j])
            )
            particles[j] += velocities[j]
            particles[j] = np.clip(particles[j], lb, ub)

        print(f"--- Iterasi {i+1} Selesai | Global Best RMSE = {global_best_score:.4f} ---\n")

        state = {
            'iterasi_terakhir': i + 1,
            'particles': particles,
            'velocities': velocities,
            'personal_best': personal_best,
            'personal_best_score': personal_best_score,
            'global_best': global_best,
            'global_best_score': global_best_score,
            'rmse_history': rmse_history
        }

        joblib.dump(state, checkpoint)
        
        if failed_count >= 5:
            print("\n[!] WARNING: 5 iterasi berturut-turut gagal! Cek rentang parameter batas bawah dan atas.")

    return rmse_history

# ==========================================
# 4. MAIN
# ==========================================
JUMLAH_PARTIKEL = 20
TARGET_ITERASI = 50
TIMEOUT_DETIK = 30 # SVR 5-Fold butuh waktu lebih lama dari single split

history = pso_auto_resume(
    n_particles=JUMLAH_PARTIKEL,
    target_iter=TARGET_ITERASI,
    timeout=TIMEOUT_DETIK
)

state = joblib.load(f"pso_state_rmse_5fold_{JUMLAH_PARTIKEL}_partikel.save")

C_best, epsilon_best, gamma_best = state['global_best']

print("\n" + "="*50)
print("HASIL AKHIR PARAMETER (Optimasi RMSE 5-Fold CV)")
print("="*50)
print(f"C      = {C_best:.4f}")
print(f"epsilon= {epsilon_best:.6f}")
print(f"gamma  = {gamma_best:.4f}")

# ==========================================
# 5. VALIDASI 10 FOLD
# ==========================================
print("\n" + "="*50)
print("VALIDASI 10-FOLD CV")
print("="*50)

best_model_cv = SVR(
    kernel='rbf',
    C=C_best,
    epsilon=epsilon_best,
    gamma=gamma_best,
    max_iter=10000
)

cv_rmse_scores = []
fold_terbaik = 1
rmse_terbaik = float('inf')

for i, (X_tr, y_tr_scaled, X_val, y_val_scaled, scaler_y_fold, y_val_asli) in enumerate(folds_data):

    # Training dengan target terskala
    best_model_cv.fit(X_tr, y_tr_scaled)

    # Prediksi
    y_pred_scaled = best_model_cv.predict(X_val)

    # Kembalikan ke skala asli (Ton)
    y_pred_asli = scaler_y_fold.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
    y_pred_asli = np.clip(y_pred_asli, 0, None)

    rmse_fold = np.sqrt(mean_squared_error(y_val_asli, y_pred_asli))
    cv_rmse_scores.append(rmse_fold)

    print(f"Fold {i+1:02d} | RMSE = {rmse_fold:.4f} Ton")

    if rmse_fold < rmse_terbaik:
        rmse_terbaik = rmse_fold
        fold_terbaik = i + 1

print(f"\nFold terbaik   : Fold {fold_terbaik}")
print(f"RMSE terbaik   : {rmse_terbaik:.4f} Ton")
print(f"RMSE rata-rata : {np.mean(cv_rmse_scores):.4f} Ton")
print(f"RMSE Std Dev   : {np.std(cv_rmse_scores):.4f} Ton")

# ==========================================
# 6. FINAL TRAINING & TESTING
# ==========================================
model_best = SVR(
    kernel='rbf',
    C=C_best,
    epsilon=epsilon_best,
    gamma=gamma_best,
    max_iter=10000
)

# Training model final
model_best.fit(X_train_final, y_train_final_scaled)

joblib.dump(model_best, 'model_svr_rbf_best_rmse_cv.save')
print('\n✓ Model final tersimpan di: model_svr_rbf_best_rmse_cv.save')

# Prediksi model
y_train_pred_scaled = model_best.predict(X_train_final)
y_test_pred_scaled = model_best.predict(X_test_final)

# Denormalisasi prediksi
y_train_pred_asli = scaler_y_final.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).ravel()
y_test_pred_asli = scaler_y_final.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()

# Cegah nilai negatif
y_train_pred_asli = np.clip(y_train_pred_asli, 0, None)
y_test_pred_asli = np.clip(y_test_pred_asli, 0, None)

y_train_asli = y_train_final_asli.ravel()
y_test_asli = y_test_final_asli.ravel()

print('\n' + '='*50)
print('EVALUASI TRAINING :')
print('='*50)
print(f"RMSE : {np.sqrt(mean_squared_error(y_train_asli, y_train_pred_asli)):.4f} Ton")
print(f"R2   : {r2_score(y_train_asli, y_train_pred_asli):.4f}")

print('\n' + '='*50)
print('EVALUASI TESTING :')
print('='*50)
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_asli, y_test_pred_asli)):.4f} Ton")
print(f"R2   : {r2_score(y_test_asli, y_test_pred_asli):.4f}")

# ==========================================
# 7. EXPORT CSV
# ==========================================
hasil = test_asli.copy()

hasil['Produksi_Asli'] = y_test_asli.round(2)
hasil['Prediksi_Ton'] = y_test_pred_asli.round(2)

kolom_kab = 'Kabupaten/Kota' if 'Kabupaten/Kota' in hasil.columns else 'Kabupaten'

if 'Tahun' in hasil.columns and 'Bulan' in hasil.columns and kolom_kab in hasil.columns:
    hasil = hasil[['Tahun', 'Bulan', kolom_kab, 'Produksi_Asli', 'Prediksi_Ton']]

nama_file_csv = f'hasil_terbaik_rmse_5fold_{JUMLAH_PARTIKEL}_partikel.csv'
hasil.to_csv(nama_file_csv, index=False)

# ==========================================
# 8. GRAFIK
# ==========================================
if len(history) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(history, linewidth=2)
    plt.title(f"Konvergensi PSO RBF ({JUMLAH_PARTIKEL} Partikel) - Optimasi 5-Fold CV", fontsize=14)
    plt.xlabel("Iterasi", fontsize=12)
    plt.ylabel("RMSE (Ton)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("\n" + "="*50)
print(">>> SELURUH PROSES SELESAI <<<")
print("="*50)